# 🐛 Cocoon defect detector — YOLOv8 training (Google Colab)

Trains a small **YOLOv8n** model that finds cocoons and labels each one **Good** or **Bad**,
then packages it for the Raspberry Pi app (`app.py`).

**Before you run anything**
1. **Runtime ▸ Change runtime type ▸ T4 GPU** (training takes only a few minutes on a GPU).
2. Have your Roboflow dataset ready, exported in **YOLOv8** format (see `README.md`, Part 2).
3. Run the cells from top to bottom (**Runtime ▸ Run all** also works — the dataset cell will ask for your input).

| Step | What happens |
|---|---|
| 1 | Install libraries |
| 2 | Get the dataset (Roboflow API *or* upload the `.zip`) |
| 3 | Check the dataset, fix `data.yaml` |
| 4 | Train (fine-tune the COCO-pretrained model — good accuracy with few epochs) |
| 5 | Measure accuracy, look at the plots |
| 6 | See predictions on your validation images |
| 7 | Export + download `cocoon_model.zip` (contains `best.pt` and the faster `best_ncnn_model/`) |

## 1 · Setup

In [ ]:
!nvidia-smi -L
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()
WORK = "/content"          # working folder (Colab)

## 2 · Get the dataset

Pick **one** way with the dropdown, then run the cell.

* **roboflow api** – no download/upload needed. In Roboflow open your dataset *version* ▸ **Export Dataset** ▸
  format **YOLOv8** ▸ **show download code** and copy the three names (workspace, project, version) and your
  **Private API key** (Roboflow ▸ Settings ▸ API Keys). The key is typed into a hidden prompt and is *not* saved in this notebook.
* **upload zip** – export as **YOLOv8 ▸ download zip to computer**, then choose that `.zip` when the file picker appears.

In [ ]:
SOURCE = "upload zip"      #@param ["upload zip", "roboflow api"]

# only used for "roboflow api" - copy these from the Roboflow "show download code" snippet
WORKSPACE = "your-workspace-id"
PROJECT   = "cocoon-defects"
VERSION   = 1

import os, zipfile
DATASET_DIR = f"{WORK}/dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

if SOURCE == "roboflow api":
    from getpass import getpass
    from roboflow import Roboflow
    rf = Roboflow(api_key=getpass("Roboflow private API key: "))
    ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8", location=DATASET_DIR)
    print("downloaded to", ds.location)
else:
    from google.colab import files
    uploaded = files.upload()                       # pick the .zip exported from Roboflow
    zip_name = next(iter(uploaded))
    with zipfile.ZipFile(zip_name) as z:
        z.extractall(DATASET_DIR)
    print("extracted", zip_name, "to", DATASET_DIR)

## 3 · Check the dataset

This cell finds `data.yaml`, rewrites it with absolute paths (Roboflow's relative `../train/images`
paths break in some setups), and prints how many images / boxes you have per split.

Things to look for: **every split has images**, **both classes have a decent number of boxes**
(aim for 30+ per class in `train`), and **no unlabeled images**.

In [ ]:
import glob, os, yaml
from collections import Counter

found = glob.glob(f"{DATASET_DIR}/**/data.yaml", recursive=True)
assert found, "No data.yaml found - was the dataset exported in YOLOv8 format?"
ROOT = os.path.dirname(found[0])
cfg = yaml.safe_load(open(found[0]))
names = cfg["names"]
names = [names[k] for k in sorted(names)] if isinstance(names, dict) else list(names)

if {n.lower() for n in names} != {"good", "bad"}:
    raise ValueError(f"Class names are {names}, but app.py expects exactly 'Good' and 'Bad'. "
                     "Rename the classes in Roboflow, make a new version and export again.")

val_dir = "valid" if os.path.isdir(f"{ROOT}/valid/images") else "val"
fixed = {"path": ROOT, "train": "train/images", "val": f"{val_dir}/images", "names": names}
if os.path.isdir(f"{ROOT}/test/images"):
    fixed["test"] = "test/images"
DATA_YAML = f"{ROOT}/data_fixed.yaml"
yaml.safe_dump(fixed, open(DATA_YAML, "w"))

print("classes:", names, "\n")
for split in ("train", val_dir, "test"):
    imgs = glob.glob(f"{ROOT}/{split}/images/*")
    if not imgs:
        continue
    counts, empty = Counter(), 0
    for lbl in glob.glob(f"{ROOT}/{split}/labels/*.txt"):
        rows = [ln for ln in open(lbl).read().splitlines() if ln.strip()]
        empty += not rows
        for ln in rows:
            counts[names[int(ln.split()[0])]] += 1
    print(f"{split:6s} {len(imgs):4d} images | boxes per class: {dict(counts)} | images without any box: {empty}")
    if split == "train" and any(counts[n] < 30 for n in names):
        print("   ⚠ fewer than 30 boxes for one class - more labeled images of that class will help a lot")
    if empty:
        print("   ⚠ images without boxes teach the model that they contain no cocoons - label them or remove them")

## 4 · Train

Why these settings give good accuracy with few epochs on a small dataset:

* **Start from COCO-pretrained weights** (`yolov8n.pt`) - the model already knows edges/shapes, it only has to learn *your* cocoons.
* **Small batch (8)** → more weight updates per epoch when there are only ~60–100 images.
* **Built-in augmentation** (mosaic, flips, small rotations, brightness/colour jitter) makes each epoch look different, so you do not need extra Roboflow augmentation.
* **Cosine learning-rate schedule + early stopping** (`patience`) - it stops by itself once validation stops improving.

`IMG_SIZE` **must be the same value in `app.py`**. 416 is a good speed/accuracy trade-off for the Raspberry Pi.

In [ ]:
from ultralytics import YOLO

MODEL_BASE = "yolov8n.pt"   #@param ["yolov8n.pt", "yolov8s.pt"]
EPOCHS     = 60             #@param {type:"integer"}
IMG_SIZE   = 416            #@param [320, 416, 480, 640] {type:"raw"}
BATCH      = 8              #@param {type:"integer"}

model = YOLO(MODEL_BASE)
results = model.train(
    data=DATA_YAML, epochs=EPOCHS, imgsz=IMG_SIZE, batch=BATCH,
    patience=20, cos_lr=True, cache=True, workers=2,
    flipud=0.5, fliplr=0.5, degrees=5, mosaic=1.0, close_mosaic=10,    # a top-down tray can be flipped freely
    project=f"{WORK}/runs", name="cocoon", exist_ok=True, plots=True, seed=0,
)
RUN_DIR = str(model.trainer.save_dir)      # folder Ultralytics used for this run (plots, weights)
BEST = str(model.trainer.best)             # the best epoch's weights
print("\nbest model:", BEST)

## 5 · How good is it?

* **mAP50** - overall detection quality. Above **0.90** is very good.
* **Precision** - of the cocoons it calls *Bad*, how many really are (low → good cocoons get pushed out).
* **Recall** - of the really *Bad* cocoons, how many it finds (low → defects slip through).
* The **confusion matrix** shows which class gets mixed up with which.

> ⚠ **Read the numbers with care.** Your photos are frames from a short session, so many are near-duplicates.
> When the validation set contains almost the same picture as the training set, the scores look better than the
> real-life accuracy. The honest test is the real machine, or photos of **different cocoons** that the model has never seen -
> put some in as a separate `test` split (see `README.md`).

In [ ]:
import os
from IPython.display import Image, display

best = YOLO(BEST)
m = best.val(data=DATA_YAML, imgsz=IMG_SIZE, split="val", plots=True, verbose=False)
print(f"mAP50 {m.box.map50:.3f}   mAP50-95 {m.box.map:.3f}   precision {m.box.mp:.3f}   recall {m.box.mr:.3f}")
for i, c in enumerate(m.box.ap_class_index):
    print(f"   {names[c]:5s} AP50 {m.box.ap50[i]:.3f}")

if "test" in fixed:
    t = best.val(data=DATA_YAML, imgsz=IMG_SIZE, split="test", verbose=False)
    print(f"TEST split: mAP50 {t.box.map50:.3f}   precision {t.box.mp:.3f}   recall {t.box.mr:.3f}")

for f in ("results.png", "confusion_matrix.png"):
    p = os.path.join(RUN_DIR, f)
    if os.path.exists(p):
        display(Image(p, width=800))

## 6 · Look at real predictions

Green = Good, red = Bad (the colour scheme of `app.py` isn't used here - Ultralytics picks its own colours, the **label text** is what matters).
Check that *both* cocoons of a pair are found and that the labels match what you see.

In [ ]:
import glob, os
import matplotlib.pyplot as plt, matplotlib.image as mpimg

val_imgs = sorted(glob.glob(f"{ROOT}/{val_dir}/images/*"))[:8]
preds = best.predict(val_imgs, imgsz=IMG_SIZE, conf=0.5, save=True, project=f"{WORK}/runs", name="preview",
                     exist_ok=True, verbose=False)
shots = sorted(glob.glob(os.path.join(str(preds[0].save_dir), "*.jpg")))[:8]
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, p in zip(axes.ravel(), shots):
    ax.imshow(mpimg.imread(p)); ax.axis("off")
plt.tight_layout(); plt.show()

### (Optional) try a photo the model has never seen

Upload one or more new photos of the tray (ideally with cocoons that are **not** in your dataset). Skip this cell if you don't have any.

In [ ]:
UPLOAD_TEST_PHOTOS = False   #@param {type:"boolean"}
if UPLOAD_TEST_PHOTOS:
    from google.colab import files
    new = files.upload()
    out = best.predict(list(new), imgsz=IMG_SIZE, conf=0.5, save=True, project=f"{WORK}/runs", name="mytest",
                       exist_ok=True, verbose=False)
    for r in out:
        display(Image(os.path.join(r.save_dir, os.path.basename(r.path)), width=640))
        print([f"{r.names[int(c)]} {p:.2f}" for c, p in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist())])

## 7 · Export + download

Creates **`cocoon_model.zip`** with

* `best.pt` – the trained model (works everywhere; ~1–2 FPS on a Raspberry Pi 4 with PyTorch),
* `best_ncnn_model/` – the same model converted to **NCNN**, which is typically several times faster on the Pi's CPU.

Copy either one next to `app.py` on the Pi (`MODEL_PATH` in `app.py`, or `python app.py --model best_ncnn_model`).
If the NCNN conversion fails here, don't worry - `best.pt` is enough; you can convert on the Pi later with
`yolo export model=best.pt format=ncnn imgsz=416`.

In [ ]:
import os, shutil

OUT = f"{WORK}/cocoon_model"
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT)
shutil.copy(BEST, f"{OUT}/best.pt")

try:
    ncnn_dir = YOLO(BEST).export(format="ncnn", imgsz=IMG_SIZE)          # returns the exported folder
    shutil.copytree(ncnn_dir, f"{OUT}/best_ncnn_model", ignore=shutil.ignore_patterns("__pycache__"))
    print("NCNN export OK ->", ncnn_dir)
except Exception as e:                                                    # optional extra, never block the download
    print("NCNN export skipped:", e)

with open(f"{OUT}/README.txt", "w") as f:
    f.write(f"classes: {names}\nimgsz: {IMG_SIZE}  <- set IMG_SIZE = {IMG_SIZE} in app.py\n")
shutil.make_archive(f"{WORK}/cocoon_model", "zip", OUT)
print("\ncontents:", sorted(os.listdir(OUT)))

try:
    from google.colab import files
    files.download(f"{WORK}/cocoon_model.zip")
except ImportError:
    print("(not running in Colab - grab the zip from", f"{WORK}/cocoon_model.zip)")

---
### Next: run it on the Raspberry Pi
1. Unzip `cocoon_model.zip`, copy `best.pt` (and/or `best_ncnn_model/`) into the same folder as `app.py`.
2. `IMG_SIZE` in `app.py` must equal the value printed above (default **416**).
3. Follow **Part 4** of `README.md` (install, wiring, first run).

*Not happy with the accuracy?* The single most effective fix is **more photos of different cocoons** (especially the class with fewer boxes),
labelled consistently - not more epochs.